In [3]:
import pandas as pd
import numpy as np

#sortowanie meczow chronologiczne aby program analizowal kazdy kolejny mecz a nie skakal miedzy roznymi meczami z roznych dat
def rankingi_elo(plik_wejsciowy, plik_wyjsciowy, plik_aktualne_elo):
    mecze = pd.read_csv(plik_wejsciowy, low_memory=False)
    mecze['Date'] = pd.to_datetime(mecze['Date'])
    mecze = mecze.sort_values(by='Date', ascending=True).reset_index(drop=True)

    wagi_turniejow = {
        'Grand Slam': 60,
        'Masters 1000': 40,
        'ATP500': 30,
        'ATP250': 20,
        'International': 20
    }

    #slowniki przechowujace elo zawodnikow
    elo_ogolne = {}
    elo_hard = {}
    elo_clay = {}
    elo_grass = {}
    elo_carpet = {}
    streaki = {}

    #kazdy gracz na poczatku dostaje 1500 punktow. wzgledem wynikow meczy wartosc elo bedzie nadpisywana
    def pobierz_elo(gracz, slownik):
        return slownik.get(gracz, 1500)

    #listy na nowe kolumny do datasetu
    W_elo_og, L_elo_og = [], []
    W_elo_h, L_elo_h = [], []
    W_elo_c, L_elo_c = [], []
    W_elo_g, L_elo_g = [], []
    W_elo_ca, L_elo_ca = [], []
    W_streak = []
    L_streak = []

    #obliczanie elo
    for index, row in mecze.iterrows():
        gracz1 = row['Player_1']
        gracz2 = row['Player_2']
        zwyciezca = row['Winner']
        nawierzchnia = row['Surface']
        turniej = row['Series']

        if zwyciezca == gracz1:
            przegrany = gracz2
        else:
            przegrany = gracz1

        K = wagi_turniejow.get(turniej, 20)
        akt_streak_W = streaki.get(zwyciezca, 0)
        akt_streak_L = streaki.get(przegrany, 0)

        #pobieranie ELO przed meczem
        w_og = pobierz_elo(zwyciezca, elo_ogolne)
        l_og = pobierz_elo(przegrany, elo_ogolne)

        w_h = pobierz_elo(zwyciezca, elo_hard)
        l_h = pobierz_elo(przegrany, elo_hard)

        w_c = pobierz_elo(zwyciezca, elo_clay)
        l_c = pobierz_elo(przegrany, elo_clay)

        w_g = pobierz_elo(zwyciezca, elo_grass)
        l_g = pobierz_elo(przegrany, elo_grass)

        w_ca = pobierz_elo(zwyciezca, elo_carpet)
        l_ca = pobierz_elo(przegrany, elo_carpet)

        #zapis do list
        W_elo_og.append(w_og); L_elo_og.append(l_og)
        W_elo_h.append(w_h); L_elo_h.append(l_h)
        W_elo_c.append(w_c); L_elo_c.append(l_c)
        W_elo_g.append(w_g); L_elo_g.append(l_g)
        W_elo_ca.append(w_ca); L_elo_ca.append(l_ca)
        W_streak.append(akt_streak_W)
        L_streak.append(akt_streak_L)

        #liczenie nowego elo ogolnego
        ocz_W = 1 / (1 + 10 ** ((l_og - w_og) / 400))
        ocz_L = 1 / (1 + 10 ** ((w_og - l_og) / 400))
        elo_ogolne[zwyciezca] = w_og + K * (1 - ocz_W)
        elo_ogolne[przegrany] = l_og + K * (0 - ocz_L)
        
        streaki[zwyciezca] = akt_streak_W + 1 if akt_streak_W >= 0 else 1
        streaki[przegrany] = akt_streak_L - 1 if akt_streak_L <= 0 else -1

        #liczenie nowego elo na danej nawierzchni
        if nawierzchnia == 'Hard':
            ocz_W = 1 / (1 + 10 ** ((l_h - w_h) / 400))
            ocz_L = 1 / (1 + 10 ** ((w_h - l_h) / 400))
            elo_hard[zwyciezca] = w_h + K * (1 - ocz_W)
            elo_hard[przegrany] = l_h + K * (0 - ocz_L)
        elif nawierzchnia == 'Clay':
            ocz_W = 1 / (1 + 10 ** ((l_c - w_c) / 400))
            ocz_L = 1 / (1 + 10 ** ((w_c - l_c) / 400))
            elo_clay[zwyciezca] = w_c + K * (1 - ocz_W)
            elo_clay[przegrany] = l_c + K * (0 - ocz_L)
        elif nawierzchnia == 'Grass':
            ocz_W = 1 / (1 + 10 ** ((l_g - w_g) / 400))
            ocz_L = 1 / (1 + 10 ** ((w_g - l_g) / 400))
            elo_grass[zwyciezca] = w_g + K * (1 - ocz_W)
            elo_grass[przegrany] = l_g + K * (0 - ocz_L)
        elif nawierzchnia == 'Carpet':
            ocz_W = 1 / (1 + 10 ** ((l_ca - w_ca) / 400))
            ocz_L = 1 / (1 + 10 ** ((w_ca - l_ca) / 400))
            elo_carpet[zwyciezca] = w_ca + K * (1 - ocz_W)
            elo_carpet[przegrany] = l_ca + K * (0 - ocz_L)

    #dodawanie kolumn do nowego pliku z rankigami
    mecze['W_ELO_ogolne'] = W_elo_og; mecze['L_ELO_ogolne'] = L_elo_og
    mecze['W_ELO_hard'] = W_elo_h; mecze['L_ELO_hard'] = L_elo_h
    mecze['W_ELO_clay'] = W_elo_c; mecze['L_ELO_clay'] = L_elo_c
    mecze['W_ELO_grass'] = W_elo_g; mecze['L_ELO_grass'] = L_elo_g
    mecze['W_ELO_carpet'] = W_elo_ca; mecze['L_ELO_carpet'] = L_elo_ca
    mecze['W_Streak'] = W_streak
    mecze['L_Streak'] = L_streak
    mecze['W_Streak_Log'] = np.sign(mecze['W_Streak']) * np.log1p(np.abs(mecze['W_Streak']))
    mecze['L_Streak_Log'] = np.sign(mecze['L_Streak']) * np.log1p(np.abs(mecze['L_Streak']))
    mecze['Roznica_Streak_Log'] = mecze['W_Streak_Log'] - mecze['L_Streak_Log']

    mecze.to_csv(plik_wyjsciowy, index=False)

    #tworzenie pliku z rankigiem elo
    wszyscy_gracze = list(elo_ogolne.keys())
    aktualne_elo =[]
    for gracz in wszyscy_gracze:
        aktualne_elo.append({
            'Player': gracz,
            'ELOogolne': elo_ogolne.get(gracz, 1500),
            'ELOhard': elo_hard.get(gracz, 1500),
            'ELOclay': elo_clay.get(gracz, 1500),
            'ELOgrass': elo_grass.get(gracz, 1500),
            'ELOcarpet': elo_carpet.get(gracz, 1500)
        })
    tabela_aktualne_elo = pd.DataFrame(aktualne_elo)
    tabela_aktualne_elo.to_csv(plik_aktualne_elo, index=False)

    return mecze, tabela_aktualne_elo


def doswiadczenie(plik_wejsciowy, plik_wyjsciowy='doswiadczenie.csv'):
    df = pd.read_csv(plik_wejsciowy, low_memory=False)
    p1 = df[['Player_1', 'Tournament']].rename(columns={'Player_1': 'player'})
    p2 = df[['Player_2', 'Tournament']].rename(columns={'Player_2': 'player'})
    
    wszystkie_wystepy = pd.concat([p1, p2], ignore_index=True)
    
    macierz_exp = pd.crosstab(wszystkie_wystepy['player'], wszystkie_wystepy['Tournament']).reset_index()
    
    macierz_exp.to_csv(plik_wyjsciowy, index=False)
    
    return macierz_exp


dane_z_elo, tabela_rankingowa = rankingi_elo('atp_tennis.csv', 'atp_tennis_z_elo.csv', 'aktualne_elo_zawodnikow.csv')
tabela_doswiadczenia = doswiadczenie('atp_tennis.csv')